In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import seaborn as sns
import torch
from bonner.datasets.allen2021_natural_scenes._stimuli import (
    download_text_annotations,
    get_coco_to_nsd_mapping,
)
from bonner.plotting import save_figure
from diffusers import DiffusionPipeline
from matplotlib import pyplot as plt
from mpl_toolkits.axes_grid1 import ImageGrid
from tqdm.auto import tqdm
from tqdm.contrib import tzip

from lib.datasets import (
    compute_shared_stimuli,
    filter_by_stimulus,
    nsd,
    split_by_repetition,
)
from lib.spectra import CrossDecomposition
from lib.utilities import JOURNAL_MATPLOTLIBRC

FIGURES_HOME = Path.cwd().parent / "figures"
FIGURES_HOME.mkdir(exist_ok=True, parents=True)

sns.set_theme(context="paper", style="ticks", rc=JOURNAL_MATPLOTLIBRC)

REFERENCE_SUBJECT = 0

stimulus_set = nsd.StimulusSet()

filepath = download_text_annotations()
with filepath.open("r") as f:
    annotations = json.load(f)

mapping = get_coco_to_nsd_mapping()
nsd_ids_in_cocotext = {
    mapping[int(image)]: len(annotations_) > 0
    for image, annotations_ in annotations["imgToAnns"].items()
    if (int(image) in mapping)
}

In [ ]:
dimensions = [1, 2, 3, 4, 5, 8, 16, 32, 256, 1024]
n_samples = 9

replace_with_synthetic = True


def z_score(x):
    return (x - x.mean()) / x.std()


if replace_with_synthetic:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    generator = torch.Generator(device=device).manual_seed(0)

    pipeline = DiffusionPipeline.from_pretrained(
        "common-canvas/CommonCanvas-XL-C",
    )
    pipeline.to(device)


rois = ("general", "V1", "faces", "places")
datasets = {
    roi: nsd.load_dataset(
        subject=REFERENCE_SUBJECT,
        roi=roi,
        preprocessing="fithrf",
        z_score=True,
    )
    for roi in rois
}

datasets = {
    roi: split_by_repetition(
        filter_by_stimulus(
            dataset,
            stimuli=compute_shared_stimuli([dataset], n_repetitions=2),
        ),
        n_repetitions=2,
    )
    for roi, dataset in datasets.items()
}

x = stimulus_set.annotations.sel(stimulus=datasets["general"][0]["stimulus"])
coords = {
    supercategory: (
        x.isel(category=x["supercategory"] == supercategory).sum("category") > 0
    ).to_numpy()
    for supercategory in ("person", "food", "food-stuff")
}
coords["food"] |= coords["food-stuff"]
coords["person"] = coords["person"].astype(np.int8)
coords["food"] = coords["food"].astype(np.int8)
text = -np.ones((len(datasets["general"][0]["stimulus"]),), dtype=np.int8)

for i_stimulus, stimulus in enumerate(
    datasets["general"][0]["stimulus"].to_numpy().tolist(),
):
    if stimulus in nsd_ids_in_cocotext:
        text[i_stimulus] = int(nsd_ids_in_cocotext[stimulus])
coords["text"] = text.astype(np.int8)


In [ ]:
palette = sns.color_palette("Set2", n_colors=3)
features = {
    "person": palette[0],
    "food": palette[1],
    "text": palette[2],
}

for roi, dataset in tqdm(datasets.items(), desc="roi"):
    # project data onto latent dimensions
    cross_decomposition = CrossDecomposition(randomized=True)
    cross_decomposition.fit(dataset[0], dataset[1])
    transformed = cross_decomposition.transform(
        dataset[0],
        direction="left",
    ).assign_coords(
        {
            "stimulus": ("presentation", dataset[0]["stimulus"].to_numpy()),
        }
        | {label: ("presentation", coord) for label, coord in coords.items()},
    )

    if False:  # FIXME
        # compute images at poles
        image_ids = {
            pole: np.empty(shape=(len(dimensions), n_samples))
            for pole in ("left", "right")
        }

        for i_dimension, dimension in enumerate(dimensions):
            selection = transformed.sel(component=dimension)
            selection = selection.sortby(selection)

            image_ids["left"][i_dimension, :] = selection[:n_samples][
                "stimulus"
            ].to_numpy()
            image_ids["right"][i_dimension, :] = selection[-n_samples:][
                "stimulus"
            ].to_numpy()

        # plot images at poles
        fig = plt.figure(figsize=(6, 6))
        subfigs = fig.subfigures(ncols=2, wspace=0.4)

        for i_pole, (pole, image_ids_) in enumerate(
            tqdm(image_ids.items(), desc="pole"),
        ):
            subfig = subfigs[i_pole]
            grid = ImageGrid(subfig, 111, nrows_ncols=(len(dimensions), n_samples))

            for image_id, ax in tzip(image_ids_.flatten(), grid, desc="image"):
                if replace_with_synthetic:
                    prompt = f"{', '.join(stimulus_set.captions[image_id])}"
                    im = (
                        pipeline(prompt, generator=generator)
                        .images[0]
                        .resize([425, 425])
                    )
                else:
                    im = stimulus_set[image_id]
                ax.imshow(im)

                ax.set_xticks([])
                ax.set_yticks([])
                ax.spines[:].set_visible(False)

            if i_pole == 0:
                for i_dimension, dimension in enumerate(dimensions):
                    ax = grid.axes_row[i_dimension][0]
                    ax.set_ylabel(dimension, rotation=0, labelpad=15, va="center")

            if i_pole == 1:
                for i_dimension, dimension in enumerate(dimensions):
                    ax = grid.axes_row[i_dimension][-1]
                    ax.yaxis.set_label_position("right")
                    ax.set_ylabel(dimension, rotation=0, labelpad=15, va="center")

        fig.suptitle(
            f"images at opposite poles of each latent dimension\nsubject 1, {roi}",
            y=0.8,
        )
        fig.set_facecolor("w")
        filename = f"dimensions-{roi}"
        if replace_with_synthetic:
            filename += "-synthetic"
        save_figure(fig, filepath=FIGURES_HOME / f"{filename}.pdf", dpi=300)

    if roi == "general":
        fig, axes = plt.subplots(figsize=(6, 7), nrows=3, sharex=True, sharey=True)
        for i_feature, (feature, color) in enumerate(features.items()):
            selections, p_values = [], []
            for component in range(1, 11):
                selection = transformed.sel(component=component).rename("score")
                selection = selection.sortby(selection)
                selection = selection.to_dataframe().assign(feature=selection[feature])
                selection["score"] = z_score(selection["score"])
                selections.append(selection)

                p_values.append(
                    scipy.stats.mannwhitneyu(
                        selection.loc[selection["feature"] == 1, "score"],
                        selection.loc[selection["feature"] == 0, "score"],
                    ).pvalue,
                )

            selections = pd.concat(selections)

            ax = axes[i_feature]
            p_values_thresholds = {
                0.0001: "***",
                0.001: "**",
                0.01: "*",
            }

            for rank, p_value in enumerate(p_values):
                for threshold, stars in p_values_thresholds.items():
                    if p_value <= threshold:
                        ax.text(
                            rank,
                            0.6,
                            stars,
                            ha="center",
                            va="center",
                            fontsize="small",
                        )
                        break

            selections["feature"] = selections["feature"].replace({1: "Yes", 0: "No"})
            sns.barplot(
                ax=ax,
                data=selections,
                x="component",
                y="score",
                hue="feature",
                hue_order=["Yes", "No"],
                palette=[color, "lightgray"],
                errorbar="se",
                width=0.4,
            )
            ax.set_ylim(bottom=-0.7, top=1.2)
            # sns.boxplot(
            #     ax=ax,
            #     data=selections,
            #     x="component",
            #     y="score",
            #     hue="feature",
            #     hue_order=["Yes", "No"],
            #     palette=[color, "lightgray"],
            #     fliersize=0,
            #     notch=True,
            #     showcaps=False,
            #     boxprops={"linewidth": 0},
            #     saturation=0.85,
            #     width=0.4,
            #     gap=0.2,
            # )
            sns.move_legend(
                ax,
                loc="upper center",
                title=f"Does the image contain {'a ' if feature == 'person' else ''}{feature}?",
                title_fontsize="medium",
                ncols=2,
                borderaxespad=0,
                borderpad=0,
            )
            ax.set_xlabel("")
            ax.set_ylabel("")
            ax.set_xticks(range(0, 10))
            ax.axhline(0, lw=0.5, c="lightgray")

            # ax.set_ylim(bottom=-5, top=6)

        fig.supxlabel("rank", ha="center", x=0.55)
        fig.supylabel("image score on latent dimension\n(Z-score)", ha="center", y=0.54)
        fig.suptitle(
            f"general visually responsive region (subject {1 + REFERENCE_SUBJECT})",
            x=0.555,
        )
        fig.set_facecolor("w")
        save_figure(fig, filepath=FIGURES_HOME / "dimensions-general-semantic.pdf")